# Notebook de ejemplo para probar la librería Trust_Library

Cargamos las librerías necesarias

In [1]:
# !pip install codecarbon
# !pip install -e .

import sys
sys.path.append("src")

import pandas as pd
import pickle
from trust_library.core import TrustEvaluator
from trust_library.factsheet import Factsheet

import matplotlib.pyplot as plt
import seaborn as sns
import json

from trust_library.utils import to_json_safe
import plotly.express as px

import numpy as np
from sklearn.model_selection import train_test_split

from UniversarSklearnWrapper import UniversalSklearnWrapper

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks

from sklearn.linear_model import LogisticRegression

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

Creamos la factsheet de ejemplo, cargamos el modelo y los datos

In [2]:
# factsheet = Factsheet.create_factsheet_interactive()
# factsheet = Factsheet(load_path="./factsheet.json")
# factsheet.save("./factsheet2.json")
factsheet = Factsheet(general={"target_column": "Target"}, fairness={"protected_feature": "Group","protected_values": [1],"favorable_outcomes": [1]}, save_path="./factsheet_actual.json")

with open("./models_and_data/model_rf.pkl", "rb") as f:
    loaded_model_rf = pickle.load(f)

train_loaded = pd.read_csv("./models_and_data/train.csv")
test_loaded = pd.read_csv("./models_and_data/test.csv")

## Uso de una métrica concreta

In [3]:
from trust_library.fairness import statistical_parity_difference

y_pred = loaded_model_rf.predict(
    test_loaded.drop(
        columns=[factsheet["general"]["target_column"]["value"]]
    )
)

group_mask = (
    test_loaded[factsheet["fairness"]["protected_feature"]["value"]]
    == factsheet["fairness"]["protected_values"]["value"][0]
)

# O bien
y_pred = loaded_model_rf.predict(test_loaded.drop("Target", axis=1))
group_mask = test_loaded["Group"] == 1

statistical_parity_difference(y_pred,group_mask)

{'value': 0.36741608418036453,
 'favored_ratio_protected': 0.5766483972762,
 'favored_ratio_unprotected': 0.20923231309583543,
 'n_protected': 6021,
 'n_unprotected': 5979,
 'n_protected_favored': 3472,
 'n_unprotected_favored': 1251}

## Hacer uso de TrustEvaluator

In [4]:
evaluator_rf = TrustEvaluator(
    model=loaded_model_rf,
    train_data=train_loaded,
    test_data=test_loaded,
    factsheet=factsheet,
    config_path="./src/trust_library/configs.json"
)

subset_rf = evaluator_rf.evaluate(show_nan=True)
# Imprimimos subset formateado y redondeamos cada valor a máximo 2 decimales
print(json.dumps(subset_rf, indent=4))

Building evaluation context...
Computing Fairness metrics...
Preparation completed in 0.00 seconds. Starting metric evaluations...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.01 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.01 s

In [5]:
evaluator_rf.print_results_summary()


RESULTADOS DE EVALUACIÓN DE TRUST

 TRUST SCORE GLOBAL: 3.38/5.0

 SCORES POR PILAR:
   - Fairness       : 3.36/5.0
   - Accountability : 2.45/5.0
   - Privacy        : 2.95/5.0
   - Sustainability : 5.00/5.0
   - Explainability : 3.57/5.0
   - Robustness     : 3.15/5.0

 MÉTRICAS DETALLADAS:

   [FAIRNESS]
      - underfitting: 5.000
      - overfitting: 2.000
      - class_balance: 1.000
      - statistical_parity_difference: 1.000
      - disparate_impact: 5.000
      - equal_opportunity_difference: 3.000
      - average_odds_difference: 4.000
      - accuracy_parity: 5.000
      - predictive_parity: 4.000
      - treatment_equality: 3.000
      - calibration_gap: 4.000
      - well_calibration_error: 5.000
      - generalized_entropy_index: 5.000
      - theil_index: 4.000
      - coefficient_of_variation: 2.000
      - individual_consistency: 4.000
      - class_imbalance: 5.000
      - kl_divergence: 2.000
      - conditional_demographic_disparity: 5.000
      - smoothed_edf: 1.

### Gráficas

In [6]:
evaluator_rf.plot_results()
evaluator_rf.plot_radar()

### Comparación entre dos modelos

In [7]:
with open("./models_and_data/model_SVM.pkl", "rb") as f:
    loaded_model_svm = pickle.load(f)
    
evaluator_svm = TrustEvaluator(loaded_model_svm, train_loaded, test_loaded, factsheet)


Building evaluation context...


In [8]:
subset_svm = evaluator_svm.evaluate(["fairness", "privacy", "robustness", "accountability", "explainability", "sustainability"], show_nan=True)
print(json.dumps(subset_svm, indent=4))

Computing Fairness metrics...
Preparation completed in 0.00 seconds. Starting metric evaluations...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.02 seconds.
Metric 'class_imbalance

In [9]:
TrustEvaluator.compare_radar({
    "RandomForest": subset_rf,
    "SVM": subset_svm
})

TrustEvaluator.compare_all_bars({
    "RandomForest": subset_rf,
    "SVM": subset_svm
})

### Modelo tree-based

In [10]:
with open("./models_and_data/model_tree.pkl", "rb") as f:
    loaded_model_tree = pickle.load(f)
evaluator_tree = TrustEvaluator(loaded_model_tree, train_loaded, test_loaded, factsheet)
subset_tree = evaluator_tree.evaluate(show_nan=True)
print(json.dumps(subset_tree, indent=4))

Building evaluation context...
Computing Fairness metrics...
Preparation completed in 0.00 seconds. Starting metric evaluations...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.02 s

### Modelo de Deep Learning - HeartDisease

In [3]:
# 1. URL DE DATOS REALES (Heart Disease Dataset)
url_data = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
df = pd.read_csv(url_data, names=columns, na_values="?")
df = df.dropna()

# Binarizar el target (0 = sin enfermedad, 1 = con enfermedad)
df["target"] = (df["target"] > 0).astype(int)

X = df.drop("target", axis=1).values
y = df["target"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. DEFINIR LA RED (Real y funcional)
class HeartDiseaseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )
    def forward(self, x):
        return self.net(x)

# 3. ENTRENAR Y GUARDAR (Simulando lo que harías en producción)
pytorch_model = HeartDiseaseNet(input_dim=X_train.shape[1])
optimizer = torch.optim.Adam(pytorch_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

for epoch in range(100): # Entrenamiento flash
    optimizer.zero_grad()
    loss = criterion(pytorch_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

# Guardamos el modelo "real"
torch.save(pytorch_model.state_dict(), "models_and_data/modelo_corazon_real.pth")
print("Modelo guardado como 'models_and_data/modelo_corazon_real.pth'")

# ==========================================
# 4. USO DEL WRAPPER DEFINITIVO CON TU LIBRERÍA
# ==========================================
# Cargas el modelo desde el archivo
modelo_cargado = HeartDiseaseNet(input_dim=X_train.shape[1])
modelo_cargado.load_state_dict(torch.load("models_and_data/modelo_corazon_real.pth"))

# Aplicas el Wrapper Definitivo
wrapped_model = UniversalSklearnWrapper(model=modelo_cargado)

Modelo guardado como 'models_and_data/modelo_corazon_real.pth'


In [ ]:
# 1. Hacemos el split manteniendo los DataFrames completos de Pandas
# (TrustEvaluator los necesita así para poder hacer drop() internamente)
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

factsheet = Factsheet(general={"target_column": "target"}, fairness={"protected_feature": "sex","protected_values": [1],"favorable_outcomes": [1]}, privacy={"quasi_identifiers": ["age", "cp"], "sensitive_attribute": ["sex"]
    })

# Asumiendo que TrustEvaluator está disponible en tu entorno
evaluator = TrustEvaluator(
    model=wrapped_model,          # <- Pasamos el modelo envuelto
    train_data=train_data,
    test_data=test_data,
    factsheet=factsheet,
    config_path="./src/trust_library/configs.json"
)

subset = evaluator.evaluate(show_nan=True)
print(json.dumps(subset, indent=4))

Building evaluation context...
Computing Fairness metrics...
Preparation completed in 0.00 seconds. Starting metric evaluations...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.01 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.00 s

In [13]:
evaluator.plot_results()

### Modelo de Deep Learning - FashionMNIST

#### Pytorch

In [14]:
# =============================================================================
# 1. DEFINICIÓN DEL MODELO CNN (Pesado)
# =============================================================================

class FashionMNISTCNN(nn.Module):
    """
    Red convolucional relativamente pesada para FashionMNIST.

    Arquitectura:
    - 3 bloques convolucionales con BatchNorm y MaxPool
    - 3 capas fully connected con Dropout
    - ~1.5M de parámetros
    """
    def __init__(self, num_classes=10):
        super(FashionMNISTCNN, self).__init__()

        # Bloque convolucional 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 28x28 -> 14x14
            nn.Dropout2d(0.25)
        )

        # Bloque convolucional 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 14x14 -> 7x7
            nn.Dropout2d(0.25)
        )

        # Bloque convolucional 3
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 7x7 -> 3x3
            nn.Dropout2d(0.25)
        )

        # Capas fully connected
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Si la entrada viene aplanada (desde sklearn), reshapearla
        if x.dim() == 2:
            # TrustEvaluator pasa columnas extra (Group, QI_1, QI_2), tomar solo los 784 píxeles
            if x.shape[1] > 784:
                x = x[:, :784]
            x = x.view(-1, 1, 28, 28)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x


# =============================================================================
# 2. FUNCIONES DE ENTRENAMIENTO
# =============================================================================

def train_model(model, train_loader, device, epochs=10, lr=0.001):
    """Entrena el modelo CNN."""
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

        scheduler.step()

        acc = 100. * correct / total
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Acc: {acc:.2f}%")

    return model


def evaluate_model(model, test_loader, device):
    """Evalúa el modelo en el conjunto de test."""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    accuracy = 100. * correct / total
    print(f"\nTest Accuracy: {accuracy:.2f}%")
    return accuracy


# =============================================================================
# 3. PREPARACIÓN DE DATOS PARA TRUST EVALUATOR
# =============================================================================

def prepare_dataframes(train_dataset, test_dataset, sample_size=5000):
    """
    Convierte los datasets de PyTorch a DataFrames de pandas para TrustEvaluator.

    TrustEvaluator espera DataFrames con:
    - Columnas de features (pixel_0, pixel_1, ..., pixel_783)
    - Una columna target
    - Columnas adicionales para fairness/privacy (Group, etc.)

    Para FashionMNIST, creamos una columna "Group" basada en si la clase es
    ropa superior (0-4) o inferior/accesorios (5-9) para las métricas de fairness.
    """
    # Extraer datos de train
    train_images = train_dataset.data.numpy()
    train_labels = train_dataset.targets.numpy()

    # Extraer datos de test
    test_images = test_dataset.data.numpy()
    test_labels = test_dataset.targets.numpy()

    # Submuestrear para acelerar la evaluación
    if sample_size and sample_size < len(train_images):
        train_indices = np.random.choice(len(train_images), sample_size, replace=False)
        train_images = train_images[train_indices]
        train_labels = train_labels[train_indices]

    if sample_size and sample_size < len(test_images):
        test_indices = np.random.choice(len(test_images), sample_size, replace=False)
        test_images = test_images[test_indices]
        test_labels = test_labels[test_indices]

    # Aplanar imágenes: (N, 28, 28) -> (N, 784)
    train_flat = train_images.reshape(len(train_images), -1)
    test_flat = test_images.reshape(len(test_images), -1)

    # Normalizar a [0, 1]
    train_flat = train_flat / 255.0
    test_flat = test_flat / 255.0

    # Crear nombres de columnas
    feature_cols = [f"pixel_{i}" for i in range(784)]

    # Crear DataFrames
    train_df = pd.DataFrame(train_flat, columns=feature_cols)
    test_df = pd.DataFrame(test_flat, columns=feature_cols)

    # Añadir columna target
    train_df["target"] = train_labels
    test_df["target"] = test_labels

    # Crear columna "Group" para fairness (0 = ropa superior, 1 = inferior/accesorios)
    # Clases FashionMNIST:
    # 0: T-shirt, 1: Trouser, 2: Pullover, 3: Dress, 4: Coat
    # 5: Sandal, 6: Shirt, 7: Sneaker, 8: Bag, 9: Ankle boot

    # Grupo 0: prendas superiores (0,2,3,4,6)
    # Grupo 1: calzado y accesorios (1,5,7,8,9)
    upper_clothes = [0, 2, 3, 4, 6]
    train_df["Group"] = train_df["target"].apply(lambda x: 0 if x in upper_clothes else 1)
    test_df["Group"] = test_df["target"].apply(lambda x: 0 if x in upper_clothes else 1)

    # Crear columnas para quasi-identificadores (usamos algunos píxeles aleatorios)
    # Esto es un ejemplo artificial para demostrar las métricas de privacy
    train_df["QI_1"] = (train_df["pixel_100"] * 10).astype(int)
    train_df["QI_2"] = (train_df["pixel_200"] * 10).astype(int)
    test_df["QI_1"] = (test_df["pixel_100"] * 10).astype(int)
    test_df["QI_2"] = (test_df["pixel_200"] * 10).astype(int)

    return train_df, test_df


In [15]:
# Configuración
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 3  # Reducido para demo, aumentar para mejor accuracy
NUM_CLASSES = 10
SAMPLE_SIZE = 3000  # Tamaño de muestra para evaluación de trust

print(f"Usando dispositivo: {DEVICE}")
print("=" * 60)

# -------------------------------------------------------------------------
# Paso 1: Descargar y preparar FashionMNIST
# -------------------------------------------------------------------------
print("\n1. Descargando FashionMNIST...")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"   Train: {len(train_dataset)} imágenes")
print(f"   Test: {len(test_dataset)} imágenes")

# -------------------------------------------------------------------------
# Paso 2: Crear y entrenar el modelo
# -------------------------------------------------------------------------
print("\n2. Creando y entrenando modelo CNN...")
print(f"   Arquitectura: FashionMNISTCNN (~1.5M parámetros)")

model = FashionMNISTCNN(num_classes=NUM_CLASSES)

try:
    model.load_state_dict(torch.load("models_and_data/fashionmnist_cnn.pth"))
    print("Modelo cargado desde models_and_data/fashionmnist_cnn.pth")
except FileNotFoundError:
    print("Archivo de modelo no encontrado. Entrenando nuevo modelo.")

    model = train_model(model, train_loader, DEVICE, epochs=EPOCHS)

    torch.save(model.state_dict(), "models_and_data/fashionmnist_cnn.pth")
    print("Modelo guardado en models_and_data/fashionmnist_cnn.pth")

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
print(f"   Total parámetros: {total_params:,}")

model = model.to(DEVICE)
model.eval()

# Evaluar
print("\n3. Evaluando modelo en test set...")
test_accuracy = evaluate_model(model, test_loader, DEVICE)


Usando dispositivo: cuda

1. Descargando FashionMNIST...
   Train: 60000 imágenes
   Test: 10000 imágenes

2. Creando y entrenando modelo CNN...
   Arquitectura: FashionMNISTCNN (~1.5M parámetros)
Modelo cargado desde models_and_data/fashionmnist_cnn.pth
   Total parámetros: 2,461,642

3. Evaluando modelo en test set...

Test Accuracy: 91.80%


#### Keras

In [5]:
# =============================================================================
# 1. DEFINICIÓN DEL MODELO CNN (Pesado) en TensorFlow/Keras
# =============================================================================

class FashionMNISTCNN(tf.keras.Model):
    """
    Red convolucional para FashionMNIST usando subclassing de Keras.
    Misma arquitectura que la versión en PyTorch.
    """
    def __init__(self, num_classes=10):
        super(FashionMNISTCNN, self).__init__()

        # Bloque convolucional 1
        self.conv1_1 = layers.Conv2D(64, (3, 3), padding='same')
        self.bn1_1 = layers.BatchNormalization()
        self.act1_1 = layers.ReLU()
        self.conv1_2 = layers.Conv2D(64, (3, 3), padding='same')
        self.bn1_2 = layers.BatchNormalization()
        self.act1_2 = layers.ReLU()
        self.pool1 = layers.MaxPooling2D((2, 2))
        self.drop1 = layers.Dropout(0.25)

        # Bloque convolucional 2
        self.conv2_1 = layers.Conv2D(128, (3, 3), padding='same')
        self.bn2_1 = layers.BatchNormalization()
        self.act2_1 = layers.ReLU()
        self.conv2_2 = layers.Conv2D(128, (3, 3), padding='same')
        self.bn2_2 = layers.BatchNormalization()
        self.act2_2 = layers.ReLU()
        self.pool2 = layers.MaxPooling2D((2, 2))
        self.drop2 = layers.Dropout(0.25)

        # Bloque convolucional 3
        self.conv3_1 = layers.Conv2D(256, (3, 3), padding='same')
        self.bn3_1 = layers.BatchNormalization()
        self.act3_1 = layers.ReLU()
        self.conv3_2 = layers.Conv2D(256, (3, 3), padding='same')
        self.bn3_2 = layers.BatchNormalization()
        self.act3_2 = layers.ReLU()
        self.pool3 = layers.MaxPooling2D((2, 2))
        self.drop3 = layers.Dropout(0.25)

        # Capas fully connected
        self.flatten = layers.Flatten()
        self.fc1 = layers.Dense(512)
        self.bn_fc1 = layers.BatchNormalization()
        self.act_fc1 = layers.ReLU()
        self.drop_fc1 = layers.Dropout(0.5)
        
        self.fc2 = layers.Dense(256)
        self.bn_fc2 = layers.BatchNormalization()
        self.act_fc2 = layers.ReLU()
        self.drop_fc2 = layers.Dropout(0.5)
        
        self.classifier = layers.Dense(num_classes)

    def call(self, inputs, training=False):
        x = inputs
        # Si la entrada viene aplanada desde tu wrapper/sklearn
        if len(x.shape) == 2:
            # Tu TrustEvaluator puede pasar columnas extra (Group, QI_1...), tomar solo 784
            if x.shape[1] > 784:
                x = x[:, :784]
            # Reshapear para TF: (Batch, Alto, Ancho, Canales) -> (N, 28, 28, 1)
            x = tf.reshape(x, [-1, 28, 28, 1])

        # Bloque 1
        x = self.conv1_1(x)
        x = self.bn1_1(x, training=training)
        x = self.act1_1(x)
        x = self.conv1_2(x)
        x = self.bn1_2(x, training=training)
        x = self.act1_2(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)

        # Bloque 2
        x = self.conv2_1(x)
        x = self.bn2_1(x, training=training)
        x = self.act2_1(x)
        x = self.conv2_2(x)
        x = self.bn2_2(x, training=training)
        x = self.act2_2(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)

        # Bloque 3
        x = self.conv3_1(x)
        x = self.bn3_1(x, training=training)
        x = self.act3_1(x)
        x = self.conv3_2(x)
        x = self.bn3_2(x, training=training)
        x = self.act3_2(x)
        x = self.pool3(x)
        x = self.drop3(x, training=training)

        # FC
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.bn_fc1(x, training=training)
        x = self.act_fc1(x)
        x = self.drop_fc1(x, training=training)
        x = self.fc2(x)
        x = self.bn_fc2(x, training=training)
        x = self.act_fc2(x)
        x = self.drop_fc2(x, training=training)

        # Keras espera logits por defecto si configuramos from_logits=True en la loss
        logits = self.classifier(x)
        return tf.nn.softmax(logits) # Forzamos salida de probabilidad

# =============================================================================
# 2. FUNCIONES DE ENTRENAMIENTO (Adaptadas a Keras)
# =============================================================================

def train_model(model, x_train, y_train, epochs=10, batch_size=64, lr=0.001):
    """Entrena el modelo CNN usando el ciclo estándar de Keras."""
    
    # Equivalente al StepLR de PyTorch
    lr_schedule = optimizers.schedules.ExponentialDecay(
        initial_learning_rate=lr,
        decay_steps=(len(x_train) // batch_size) * 5, # Reduce cada 5 epochs
        decay_rate=0.5,
        staircase=True
    )
    
    optimizer = optimizers.Adam(learning_rate=lr_schedule)
    # Importante: from_logits=False porque la última capa tiene Softmax
    loss_fn = losses.SparseCategoricalCrossentropy(from_logits=False)

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

    # En TensorFlow llamamos directamente al método fit
    model.fit(
        x_train, 
        y_train, 
        epochs=epochs, 
        batch_size=batch_size, 
        verbose=1
    )
    return model

def evaluate_model(model, x_test, y_test):
    """Evalúa el modelo en el conjunto de test."""
    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
    print(f"\nTest Accuracy: {accuracy * 100:.2f}%")
    return accuracy


# =============================================================================
# 3. PREPARACIÓN DE DATOS PARA TRUST EVALUATOR (Directo desde Keras)
# =============================================================================

def prepare_dataframes(sample_size=5000):
    """
    Descarga los datos directamente desde tf.keras.datasets.
    La lógica de Pandas se mantiene idéntica para que encaje con el Factsheet.
    """
    (train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.fashion_mnist.load_data()

    if sample_size and sample_size < len(train_images):
        train_indices = np.random.choice(len(train_images), sample_size, replace=False)
        train_images = train_images[train_indices]
        train_labels = train_labels[train_indices]

    if sample_size and sample_size < len(test_images):
        test_indices = np.random.choice(len(test_images), sample_size, replace=False)
        test_images = test_images[test_indices]
        test_labels = test_labels[test_indices]

    # Aplanar imágenes: (N, 28, 28) -> (N, 784)
    train_flat = train_images.reshape(len(train_images), -1)
    test_flat = test_images.reshape(len(test_images), -1)

    # Normalizar a [0, 1] (Convertimos a float32, estándar en TF)
    train_flat = train_flat.astype("float32") / 255.0
    test_flat = test_flat.astype("float32") / 255.0

    feature_cols = [f"pixel_{i}" for i in range(784)]
    train_df = pd.DataFrame(train_flat, columns=feature_cols)
    test_df = pd.DataFrame(test_flat, columns=feature_cols)

    train_df["target"] = train_labels
    test_df["target"] = test_labels

    upper_clothes = [0, 2, 3, 4, 6]
    train_df["Group"] = train_df["target"].apply(lambda x: 0 if x in upper_clothes else 1)
    test_df["Group"] = test_df["target"].apply(lambda x: 0 if x in upper_clothes else 1)

    train_df["QI_1"] = (train_df["pixel_100"] * 10).astype(int)
    train_df["QI_2"] = (train_df["pixel_200"] * 10).astype(int)
    test_df["QI_1"] = (test_df["pixel_100"] * 10).astype(int)
    test_df["QI_2"] = (test_df["pixel_200"] * 10).astype(int)

    return train_df, test_df

In [6]:
# Configuración
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 3  # Reducido para demo, aumentar para mejor accuracy
NUM_CLASSES = 10
SAMPLE_SIZE = 3000  # Tamaño de muestra para evaluación de trust

print(f"Usando dispositivo: {DEVICE}")
print("=" * 60)
# -------------------------------------------------------------------------
# Paso 1: Descargar y preparar FashionMNIST (Versión TensorFlow)
# -------------------------------------------------------------------------
print("\n1. Descargando FashionMNIST...")

# Keras ya incluye el dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Normalizamos a flotantes entre 0 y 1 y añadimos la dimensión del canal
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# TensorFlow espera formato (Batch, Height, Width, Channels)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print(f"   Train: {len(x_train)} imágenes")
print(f"   Test: {len(x_test)} imágenes")

# -------------------------------------------------------------------------
# Paso 2: Crear y entrenar el modelo (Versión TensorFlow)
# -------------------------------------------------------------------------
print("\n2. Creando y entrenando modelo CNN...")
print(f"   Arquitectura: FashionMNISTCNN (TensorFlow)")

model = FashionMNISTCNN(num_classes=NUM_CLASSES)

# Llamamos a la función train_model que diseñamos para TF
# Nota: pasamos x_train, y_train directamente, sin DataLoader
model = train_model(model, x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

# Contar parámetros
total_params = model.count_params()
print(f"   Total parámetros: {total_params:,}")

# Evaluar
print("\n3. Evaluando modelo en test set...")
test_accuracy = evaluate_model(model, x_test, y_test)

Usando dispositivo: cuda

1. Descargando FashionMNIST...
   Train: 60000 imágenes
   Test: 10000 imágenes

2. Creando y entrenando modelo CNN...
   Arquitectura: FashionMNISTCNN (TensorFlow)
Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 84s 173ms/step - accuracy: 0.8019 - loss: 0.5544
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 83s 178ms/step - accuracy: 0.8866 - loss: 0.3156
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 85s 181ms/step - accuracy: 0.9045 - loss: 0.2684
   Total parámetros: 2,464,970

3. Evaluando modelo en test set...

Test Accuracy: 90.93%


In [7]:
# -------------------------------------------------------------------------
# Paso 3: Preparar datos para TrustEvaluator
# -------------------------------------------------------------------------
print(f"\n4. Preparando DataFrames (muestra de {SAMPLE_SIZE} para trust)...")

# Recargar datasets sin transformaciones para obtener pixeles crudos
# raw_train = datasets.FashionMNIST(root='./data', train=True, download=False)
# raw_test = datasets.FashionMNIST(root='./data', train=False, download=False)

# train_df, test_df = prepare_dataframes(raw_train, raw_test, sample_size=SAMPLE_SIZE)
train_df, test_df = prepare_dataframes(sample_size=SAMPLE_SIZE)

print(f"   Train DataFrame: {train_df.shape}")
print(f"   Test DataFrame: {test_df.shape}")

# -------------------------------------------------------------------------
# Paso 4: Crear Factsheet
# -------------------------------------------------------------------------
print("\n5. Creando Factsheet...")

factsheet = Factsheet(
    general={
        "model_name": "FashionMNIST-CNN",
        "model_type": "DeepLearning",
        "purpose_description": "Clasificación de imágenes de prendas de vestir",
        "domain_description": "Visión por computador / Retail",
        "training_data_description": "FashionMNIST: 60k imágenes 28x28 en escala de grises de 10 categorías de ropa",
        "model_information": f"CNN con 3 bloques conv + 3 FC, {total_params:,} parámetros, entrenado {EPOCHS} epochs",
        "target_column": "target",
        "authors": ["Demo User"],
        "contact_information": "demo@example.com"
    },
    fairness={
        "protected_feature": "Group",
        "protected_values": [1],  # Grupo de calzado/accesorios como grupo desprotegido
        "favorable_outcomes": [0, 1, 2, 3, 4]  # Clasificación correcta en ropa superior
    },
    privacy={
        "epsilon": 1.0,
        "quasi_identifiers": ["QI_1", "QI_2"],
        "sensitive_attribute": ["Group"]
    },
    sustainability={
        "use_codecarbon": False,
        "energy_consumed": 0.01,  # Estimación para demo
        "cpu_energy": 0.008,
        "gpu_energy": 0.002 if torch.cuda.is_available() else 0.0,
        "ram_energy": 0.001,
        "emissions": 0.005,
        "duration": EPOCHS * 60,  # Aproximación
        "country": "Spain",
        "pue": 1.0,
        "wue": 0.0
    }
)

print("   Factsheet guardada en: factsheet_fashionmnist.json")

# -------------------------------------------------------------------------
# Paso 5: Envolver modelo con UniversalSklearnWrapper
# -------------------------------------------------------------------------
print("\n6. Envolviendo modelo PyTorch con UniversalSklearnWrapper...")

wrapped_model = UniversalSklearnWrapper(
    model=model,
    device=str(DEVICE),
    num_classes=NUM_CLASSES
)

print(f"   Framework detectado: {wrapped_model.framework}")

# -------------------------------------------------------------------------
# Paso 6: Evaluar con TrustEvaluator
# -------------------------------------------------------------------------
print("\n7. Ejecutando TrustEvaluator...")
print("=" * 60)

evaluator = TrustEvaluator(
    model=wrapped_model,
    train_data=train_df,
    test_data=test_df,
    factsheet=factsheet,
    output_path="trust_evaluation_fashionmnist.json"
)

# Evaluar todos los pilares
subset_keras = evaluator.evaluate(show_nan=True)

# evaluator.plot_results()
# evaluator.plot_radar()

print(json.dumps(subset_keras, indent=4))


4. Preparando DataFrames (muestra de 3000 para trust)...
   Train DataFrame: (3000, 788)
   Test DataFrame: (3000, 788)

5. Creando Factsheet...
   Factsheet guardada en: factsheet_fashionmnist.json

6. Envolviendo modelo PyTorch con UniversalSklearnWrapper...
   Framework detectado: tensorflow

7. Ejecutando TrustEvaluator...
Building evaluation context...
Computing Fairness metrics...
Preparation completed in 0.00 seconds. Starting metric evaluations...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 second

### Modelo no sesgado vs sesgado. Dataset sintético

In [ ]:
# Fijamos semilla para reproducibilidad
np.random.seed(42)
n_samples = 3000

# =====================================================================
# ESCENARIO 1: MUY SESGADO (Extrema dependencia del atributo sensible)
# =====================================================================
# Simulamos dos features genéricas (ej. años de experiencia, puntuación test)
X_biased_f1 = np.random.normal(50, 10, n_samples)
X_biased_f2 = np.random.normal(50, 10, n_samples)

# Atributo sensible: 0 (Hombres), 1 (Mujeres)
sex_biased = np.random.binomial(1, 0.5, n_samples)

# CREACIÓN DEL SESGO: Si es Hombre (0) tiene 95% de probabilidad de éxito.
# Si es Mujer (1) tiene solo un 5% de probabilidad, ignorando su mérito (f1, f2).
probs_biased = np.where(sex_biased == 0, 0.95, 0.05)
income_biased = np.random.binomial(1, probs_biased)

df_biased = pd.DataFrame({'feature1': X_biased_f1, 'feature2': X_biased_f2, 'sex': sex_biased, 'income': income_biased})
train_biased, test_biased = train_test_split(df_biased, test_size=0.3, random_state=42)

# Entrenar modelo (Aprenderá rápidamente a usar 'sex' para predecir)
X_train_b = train_biased.drop(columns=['income'])
y_train_b = train_biased['income']
X_test_b  = test_biased.drop(columns=['income'])

model_biased = LogisticRegression()
model_biased.fit(X_train_b, y_train_b)


# =====================================================================
# ESCENARIO 2: NADA SESGADO (Independencia total del atributo sensible)
# =====================================================================
X_fair_f1 = np.random.normal(50, 10, n_samples)
X_fair_f2 = np.random.normal(50, 10, n_samples)
sex_fair = np.random.binomial(1, 0.5, n_samples)

# CREACIÓN JUSTA: El éxito ('income') depende estrictamente de sumar feature1 y feature2.
# El sexo (0 o 1) no tiene NINGÚN peso en el resultado.
score = X_fair_f1 + X_fair_f2
# Usamos una función sigmoide para convertir el score en probabilidades
probs_fair = 1 / (1 + np.exp(-(score - 100) / 5)) 
income_fair = np.random.binomial(1, probs_fair)

df_fair = pd.DataFrame({'feature1': X_fair_f1, 'feature2': X_fair_f2, 'sex': sex_fair, 'income': income_fair})
train_fair, test_fair = train_test_split(df_fair, test_size=0.3, random_state=42)

# Entrenar modelo (Aprenderá los pesos de las features, ignorará el sexo)
X_train_f = train_fair.drop(columns=['income'])
y_train_f = train_fair['income']
X_test_f  = test_fair.drop(columns=['income'])

model_fair = LogisticRegression()
model_fair.fit(X_train_f, y_train_f)

# =====================================================================
# EVALUACIÓN (Integración con tu código)
# =====================================================================
# 1. Configurar Factsheet (igual para ambos, adaptado a los nuevos nombres)
# Asumiendo que has instanciado factsheet2 = Factsheet()
factsheet2 = Factsheet()

factsheet2["general"]["target_column"]["value"] = "income"
factsheet2["fairness"]["protected_feature"]["value"] = "sex"
factsheet2["fairness"]["protected_values"]["value"] = [1] 
factsheet2["fairness"]["favorable_outcomes"]["value"] = [1]
factsheet2["privacy"]["quasi_identifiers"]["value"] = ["feature1", "feature2"]
factsheet2["privacy"]["sensitive_attribute"]["value"] = ["sex"]
# Omitimos privacy_identifiers ya que son datos sintéticos

# 2. Evaluar Escenario Biased
evaluator_biased = TrustEvaluator(
    model=model_biased,
    train_data=train_biased,
    test_data=test_biased,
    factsheet=factsheet2
)
subset_biased = evaluator_biased.evaluate(show_nan=True)

# 3. Evaluar Escenario Fair
evaluator_fair = TrustEvaluator(
    model=model_fair,
    train_data=train_fair,
    test_data=test_fair,
    factsheet=factsheet2
)
subset_fair = evaluator_fair.evaluate(show_nan=True)

print(json.dumps(subset_fair, indent=4))

# 4. Comparar (Tus métricas deberían mostrar diferencias abismales aquí)
TrustEvaluator.compare_radar({
    "Unbiased": subset_fair,
    "Highly Biased": subset_biased
})

TrustEvaluator.compare_all_bars({
    "Normal": subset_fair,
    "Biased": subset_biased
})

Building evaluation context...
Computing Fairness metrics...
Preparation completed in 0.00 seconds. Starting metric evaluations...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.01 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.01 s

### Regresión

In [20]:
"""
Script para probar TrustEvaluator con un modelo de REGRESIÓN.

Dataset: California Housing (sklearn)
- Predecir el valor medio de casas en California

Este script:
1. Descarga el dataset California Housing
2. Preprocesa los datos
3. Entrena un modelo GradientBoosting Regressor
4. Crea la Factsheet correspondiente
5. Evalúa el modelo con TrustEvaluator
"""

# =============================================================================
# 1. CARGA Y PREPARACIÓN DE DATOS
# =============================================================================

def load_and_prepare_data():
    """
    Carga el dataset California Housing y lo prepara para regresión.

    Returns:
        train_df, test_df: DataFrames con features y target
        feature_names: Lista de nombres de features
    """
    print("Descargando California Housing dataset...")

    # Cargar dataset
    housing = fetch_california_housing()

    # Crear DataFrame
    df = pd.DataFrame(housing.data, columns=housing.feature_names)
    df['target'] = housing.target  # Valor medio de casas (en $100,000)

    print(f"   Dataset: {len(df)} registros")
    print(f"   Features: {len(housing.feature_names)}")
    print(f"\n   Descripción de features:")
    for name, desc in zip(housing.feature_names, [
        "Ingreso medio del bloque",
        "Edad media de las casas",
        "Número medio de habitaciones",
        "Número medio de dormitorios",
        "Población del bloque",
        "Ocupación media",
        "Latitud",
        "Longitud"
    ]):
        print(f"      - {name}: {desc}")

    print(f"\n   Target: Valor medio de casas ($100,000)")
    print(f"   Rango target: [{df['target'].min():.2f}, {df['target'].max():.2f}]")

    # Crear columna "Group" para métricas de fairness
    # Basado en ingreso medio: bajo (0) vs alto (1)
    median_income = df['MedInc'].median()
    df['Group'] = (df['MedInc'] > median_income).astype(int)

    # Crear quasi-identificadores para privacy
    # Basados en ubicación geográfica discretizada
    df['QI_lat'] = pd.qcut(df['Latitude'], q=10, labels=False, duplicates='drop')
    df['QI_lon'] = pd.qcut(df['Longitude'], q=10, labels=False, duplicates='drop')

    # Separar train/test
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

    print(f"\n   Train: {len(train_df)} registros")
    print(f"   Test: {len(test_df)} registros")

    feature_names = list(housing.feature_names)

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), feature_names


def get_feature_columns(df):
    """Obtiene las columnas de features (excluyendo target y metadatos)."""
    exclude = ['target', 'Group', 'QI_lat', 'QI_lon']
    return [c for c in df.columns if c not in exclude]


# =============================================================================
# 2. ENTRENAMIENTO
# =============================================================================

def train_model(train_df, feature_cols):
    """Entrena un modelo GradientBoosting Regressor."""
    X_train = train_df[feature_cols]
    y_train = train_df['target']

    # Escalar features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # Entrenar GradientBoosting
    print("\nEntrenando GradientBoosting Regressor...")
    model = GradientBoostingRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        min_samples_split=5,
        min_samples_leaf=3,
        subsample=0.8,
        random_state=42,
        verbose=0
    )

    model.fit(X_train_scaled, y_train)

    # Guardar scaler junto con el modelo
    model.scaler_ = scaler
    model.feature_cols_ = feature_cols

    return model


def evaluate_model(model, test_df, feature_cols):
    """Evalúa el modelo en el conjunto de test."""
    X_test = test_df[feature_cols]
    y_test = test_df['target']

    X_test_scaled = model.scaler_.transform(X_test)
    y_pred = model.predict(X_test_scaled)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"\nMétricas de Test:")
    print(f"   MSE:  {mse:.4f}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAE:  {mae:.4f}")
    print(f"   R²:   {r2:.4f}")

    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}


# =============================================================================
# 3. WRAPPER PARA TRUST EVALUATOR
# =============================================================================

class ScaledRegressorWrapper:
    """
    Wrapper que aplica el scaler antes de predecir.
    Necesario porque TrustEvaluator pasa datos sin escalar.
    """
    def __init__(self, model, scaler, feature_cols):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        # Para regresión no hay classes_
        self._is_regressor = True

    def predict(self, X):
        # Filtrar solo las columnas de features
        if isinstance(X, pd.DataFrame):
            X_features = X[self.feature_cols].values
        else:
            X_features = X[:, :len(self.feature_cols)]

        X_scaled = self.scaler.transform(X_features)
        return self.model.predict(X_scaled)

    # Regresores no tienen predict_proba
    # pero podemos definirlo para evitar errores
    def predict_proba(self, X):
        return None


In [21]:
print("=" * 70)
print("EVALUACIÓN DE TRUST - REGRESIÓN (CALIFORNIA HOUSING)")
print("=" * 70)

# -------------------------------------------------------------------------
# Paso 1: Cargar y preparar datos
# -------------------------------------------------------------------------
print("\n1. Cargando dataset California Housing...")
train_df, test_df, feature_names = load_and_prepare_data()
feature_cols = get_feature_columns(train_df)

# -------------------------------------------------------------------------
# Paso 2: Entrenar modelo
# -------------------------------------------------------------------------
print("\n2. Entrenando modelo...")
model = train_model(train_df, feature_cols)

# Feature importances
print("\n   Importancia de features:")
importances = model.feature_importances_
for name, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
    print(f"      {name:15s}: {imp:.4f}")

# -------------------------------------------------------------------------
# Paso 3: Evaluar modelo
# -------------------------------------------------------------------------
print("\n3. Evaluando modelo...")
metrics = evaluate_model(model, test_df, feature_cols)

# Guardar modelo
joblib.dump(model, 'models_and_data/model_regression.pkl')
print("\n   Modelo guardado en: models_and_data/model_regression.pkl")

# -------------------------------------------------------------------------
# Paso 4: Crear Factsheet
# -------------------------------------------------------------------------
print("\n4. Creando Factsheet...")

factsheet = Factsheet(
    general={
        "model_name": "CaliforniaHousing-Regressor",
        "model_type": "GradientBoostingRegressor",
        "purpose_description": "Predicción del valor medio de viviendas en California basado en características demográficas y geográficas",
        "domain_description": "Inmobiliario / Valuación de propiedades",
        "training_data_description": "California Housing Dataset: 20,640 bloques censales con 8 features demográficas y geográficas",
        "model_information": f"GradientBoosting con 200 estimadores, max_depth=5, lr=0.1, {len(feature_cols)} features",
        "target_column": "target",
        "authors": ["Data Science Team"],
        "contact_information": "datascience@example.com"
    },
    fairness={
        "protected_feature": "Group",
        "protected_values": [0],  # Bajo ingreso como grupo desprotegido
        "favorable_outcomes": [1]  # Valor alto de casa como favorable (para métricas)
    },
    privacy={
        "epsilon": 1.0,
        "quasi_identifiers": ["QI_lat", "QI_lon"],
        "sensitive_attribute": ["Group"]
    },
    sustainability={
        "use_codecarbon": False,
        "energy_consumed": 0.003,
        "cpu_energy": 0.002,
        "gpu_energy": 0.0,
        "ram_energy": 0.001,
        "emissions": 0.001,
        "duration": 20,
        "country": "Spain",
        "pue": 1.0,
        "wue": 0.0
    },
    save_path="models_and_data/factsheet_regression.json"
)

print("   Factsheet guardada en: models_and_data/factsheet_regression.json")

# -------------------------------------------------------------------------
# Paso 5: Crear wrapper y evaluar con TrustEvaluator
# -------------------------------------------------------------------------
print("\n5. Evaluando con TrustEvaluator...")
print("=" * 70)

wrapped_model = ScaledRegressorWrapper(model, model.scaler_, feature_cols)

evaluator = TrustEvaluator(
    model=wrapped_model,
    train_data=train_df,
    test_data=test_df,
    factsheet=factsheet,
    output_path="models_and_data/trust_evaluation_regression.json"
)

subset_multi = evaluator.evaluate(show_nan=True)


# # Información adicional sobre regresión
# print("\n MÉTRICAS DE REGRESIÓN:")
# print(f"   - MSE:  {metrics['mse']:.4f}")
# print(f"   - RMSE: {metrics['rmse']:.4f}")
# print(f"   - MAE:  {metrics['mae']:.4f}")
# print(f"   - R²:   {metrics['r2']:.4f}")

# Visualizaciones
# evaluator.plot_results()
# evaluator.plot_radar()

print(json.dumps(subset_multi, indent=4))

EVALUACIÓN DE TRUST - REGRESIÓN (CALIFORNIA HOUSING)

1. Cargando dataset California Housing...
Descargando California Housing dataset...
   Dataset: 20640 registros
   Features: 8

   Descripción de features:
      - MedInc: Ingreso medio del bloque
      - HouseAge: Edad media de las casas
      - AveRooms: Número medio de habitaciones
      - AveBedrms: Número medio de dormitorios
      - Population: Población del bloque
      - AveOccup: Ocupación media
      - Latitude: Latitud
      - Longitude: Longitud

   Target: Valor medio de casas ($100,000)
   Rango target: [0.15, 5.00]

   Train: 16512 registros
   Test: 4128 registros

2. Entrenando modelo...

Entrenando GradientBoosting Regressor...

   Importancia de features:
      MedInc         : 0.5462
      AveOccup       : 0.1354
      Longitude      : 0.1073
      Latitude       : 0.1060
      HouseAge       : 0.0446
      AveRooms       : 0.0317
      AveBedrms      : 0.0146
      Population     : 0.0142

3. Evaluando modelo...

### Multiclase

In [22]:
"""
Script para probar TrustEvaluator con un modelo de clasificación multiclase
para detección de ataques en tráfico de red (ciberseguridad).

Dataset: KDD Cup 1999 (versión reducida via sklearn)
- Tráfico normal + 9 tipos de ataques agrupados

Este script:
1. Descarga el dataset KDD Cup 99
2. Preprocesa y agrupa los ataques en categorías
3. Entrena un modelo RandomForest
4. Crea la Factsheet correspondiente
5. Evalúa el modelo con TrustEvaluator
"""

# =============================================================================
# 1. MAPEO DE ATAQUES
# =============================================================================

# Mapeo de ataques específicos a categorías generales
ATTACK_MAPPING = {
    # Normal
    b'normal.': 'normal',

    # DoS - Denial of Service
    b'back.': 'dos',
    b'land.': 'dos',
    b'neptune.': 'dos',
    b'pod.': 'dos',
    b'smurf.': 'dos',
    b'teardrop.': 'dos',
    b'apache2.': 'dos',
    b'udpstorm.': 'dos',
    b'processtable.': 'dos',
    b'mailbomb.': 'dos',

    # Probe - Surveillance/Scanning
    b'satan.': 'probe',
    b'ipsweep.': 'probe',
    b'nmap.': 'probe',
    b'portsweep.': 'probe',
    b'mscan.': 'probe',
    b'saint.': 'probe',

    # R2L - Remote to Local
    b'guess_passwd.': 'r2l',
    b'ftp_write.': 'r2l',
    b'imap.': 'r2l',
    b'phf.': 'r2l',
    b'multihop.': 'r2l',
    b'warezmaster.': 'r2l',
    b'warezclient.': 'r2l',
    b'spy.': 'r2l',
    b'xlock.': 'r2l',
    b'xsnoop.': 'r2l',
    b'snmpguess.': 'r2l',
    b'snmpgetattack.': 'r2l',
    b'httptunnel.': 'r2l',
    b'sendmail.': 'r2l',
    b'named.': 'r2l',
    b'worm.': 'r2l',

    # U2R - User to Root (Privilege Escalation)
    b'buffer_overflow.': 'u2r',
    b'loadmodule.': 'u2r',
    b'rootkit.': 'u2r',
    b'perl.': 'u2r',
    b'sqlattack.': 'u2r',
    b'xterm.': 'u2r',
    b'ps.': 'u2r',

    # Specific attack subtypes for more granularity
    b'portsweep.': 'portsweep',
    b'nmap.': 'nmap_scan',
    b'neptune.': 'neptune_dos',
    b'smurf.': 'smurf_dos',
}

# Versión simplificada: 10 clases (normal + 9 tipos de ataques)
ATTACK_CATEGORIES = {
    b'normal.': 0,      # Normal traffic
    b'neptune.': 1,     # Neptune DoS
    b'smurf.': 2,       # Smurf DoS
    b'back.': 3,        # Back DoS
    b'teardrop.': 4,    # Teardrop DoS
    b'satan.': 5,       # Satan Probe
    b'ipsweep.': 6,     # IP Sweep Probe
    b'portsweep.': 7,   # Port Sweep Probe
    b'nmap.': 8,        # Nmap Scan
    b'warezclient.': 9, # Warez Client R2L
}

CLASS_NAMES = [
    'normal', 'neptune_dos', 'smurf_dos', 'back_dos', 'teardrop_dos',
    'satan_probe', 'ipsweep_probe', 'portsweep_probe', 'nmap_scan', 'warezclient_r2l'
]


# =============================================================================
# 2. FUNCIONES DE PREPARACIÓN DE DATOS
# =============================================================================

def load_and_prepare_data(sample_size=50000):
    """
    Carga el dataset KDD Cup 99 y lo prepara para clasificación multiclase.

    Returns:
        train_df, test_df: DataFrames con features y target
        feature_names: Lista de nombres de features
    """
    print("Descargando KDD Cup 99 dataset...")

    # Cargar dataset (versión 10%)
    kdd = fetch_kddcup99(subset='SA', percent10=True, random_state=42)

    # Nombres de features del KDD Cup 99
    feature_names = [
        'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
        'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
        'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
        'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
        'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
        'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
        'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
        'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
        'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
        'dst_host_rerror_rate', 'dst_host_srv_rerror_rate'
    ]

    # Crear DataFrame
    df = pd.DataFrame(kdd.data, columns=feature_names)
    df['attack_type'] = kdd.target

    print(f"   Dataset original: {len(df)} registros")

    # Filtrar solo las clases que queremos (las 10 más comunes)
    df = df[df['attack_type'].isin(ATTACK_CATEGORIES.keys())]
    print(f"   Después de filtrar clases: {len(df)} registros")

    # Mapear a categorías numéricas
    df['target'] = df['attack_type'].map(ATTACK_CATEGORIES)
    df = df.drop(columns=['attack_type'])

    # Submuestrear si es necesario
    if sample_size and len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42)
        print(f"   Después de submuestrear: {len(df)} registros")

    # Filtrar clases con muy pocos ejemplos (mínimo 10 para poder estratificar)
    class_counts = df['target'].value_counts()
    valid_classes = class_counts[class_counts >= 10].index.tolist()
    df = df[df['target'].isin(valid_classes)]

    # Remapear clases a valores consecutivos
    old_to_new = {old: new for new, old in enumerate(sorted(valid_classes))}
    df['target'] = df['target'].map(old_to_new)

    # Actualizar nombres de clases usados
    global CLASS_NAMES
    CLASS_NAMES = [CLASS_NAMES[i] for i in sorted(valid_classes)]

    print(f"   Después de filtrar clases raras: {len(df)} registros")

    # Codificar variables categóricas
    categorical_cols = ['protocol_type', 'service', 'flag']
    label_encoders = {}

    for col in categorical_cols:
        le = LabelEncoder()
        # Convertir bytes a string si es necesario
        df[col] = df[col].apply(lambda x: x.decode() if isinstance(x, bytes) else x)
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

    # Crear columna "Group" para métricas de fairness
    # Grupo 0: Tráfico normal, Grupo 1: Ataques
    df['Group'] = (df['target'] != 0).astype(int)

    # Crear quasi-identificadores para privacy
    df['QI_src_bytes'] = pd.qcut(df['src_bytes'], q=10, labels=False, duplicates='drop')
    df['QI_dst_bytes'] = pd.qcut(df['dst_bytes'], q=10, labels=False, duplicates='drop')

    # Separar train/test
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

    print(f"\n   Train: {len(train_df)} registros")
    print(f"   Test: {len(test_df)} registros")
    print(f"   Clases: {df['target'].nunique()}")
    print(f"   Distribución de clases:")
    for i, name in enumerate(CLASS_NAMES):
        count = (df['target'] == i).sum()
        print(f"      {i}: {name:20s} - {count:6d} ({100*count/len(df):.1f}%)")

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), feature_names


def get_feature_columns(df):
    """Obtiene las columnas de features (excluyendo target y metadatos)."""
    exclude = ['target', 'Group', 'QI_src_bytes', 'QI_dst_bytes']
    return [c for c in df.columns if c not in exclude]


# =============================================================================
# 3. ENTRENAMIENTO
# =============================================================================

def train_model(train_df, feature_cols):
    """Entrena un modelo RandomForest para clasificación multiclase."""
    X_train = train_df[feature_cols]
    y_train = train_df['target']

    # Escalar features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # Entrenar RandomForest
    print("\nEntrenando RandomForest...")
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42,
        class_weight='balanced'
    )

    model.fit(X_train_scaled, y_train)

    # Guardar scaler junto con el modelo
    model.scaler_ = scaler
    model.feature_cols_ = feature_cols

    return model


def evaluate_model(model, test_df, feature_cols):
    """Evalúa el modelo en el conjunto de test."""
    X_test = test_df[feature_cols]
    y_test = test_df['target']

    X_test_scaled = model.scaler_.transform(X_test)
    y_pred = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nTest Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")

    # Obtener las clases presentes en los datos
    present_classes = sorted(y_test.unique())
    present_names = [CLASS_NAMES[i] for i in present_classes if i < len(CLASS_NAMES)]

    print(classification_report(y_test, y_pred, labels=present_classes,
                                target_names=present_names, zero_division=0))

    return accuracy


# =============================================================================
# 4. WRAPPER PARA TRUST EVALUATOR
# =============================================================================

class ScaledModelWrapper:
    """
    Wrapper que aplica el scaler antes de predecir.
    Necesario porque TrustEvaluator pasa datos sin escalar.
    """
    def __init__(self, model, scaler, feature_cols):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.classes_ = model.classes_

    def predict(self, X):
        # Filtrar solo las columnas de features
        if isinstance(X, pd.DataFrame):
            X_features = X[self.feature_cols].values
        else:
            X_features = X[:, :len(self.feature_cols)]

        X_scaled = self.scaler.transform(X_features)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X_features = X[self.feature_cols].values
        else:
            X_features = X[:, :len(self.feature_cols)]

        X_scaled = self.scaler.transform(X_features)
        return self.model.predict_proba(X_scaled)

In [23]:
print("=" * 70)
print("EVALUACIÓN DE TRUST - CLASIFICACIÓN MULTICLASE DE ATAQUES DE RED")
print("=" * 70)

# -------------------------------------------------------------------------
# Paso 1: Cargar y preparar datos
# -------------------------------------------------------------------------
print("\n1. Cargando dataset KDD Cup 99...")
train_df, test_df, feature_names = load_and_prepare_data(sample_size=30000)
feature_cols = get_feature_columns(train_df)

# -------------------------------------------------------------------------
# Paso 2: Entrenar modelo
# -------------------------------------------------------------------------
print("\n2. Entrenando modelo...")
model = train_model(train_df, feature_cols)

# -------------------------------------------------------------------------
# Paso 3: Evaluar modelo
# -------------------------------------------------------------------------
print("\n3. Evaluando modelo...")
accuracy = evaluate_model(model, test_df, feature_cols)

# Guardar modelo
joblib.dump(model, 'models_and_data/model_cybersecurity.pkl')
print("\n   Modelo guardado en: models_and_data/model_cybersecurity.pkl")

# -------------------------------------------------------------------------
# Paso 4: Crear Factsheet
# -------------------------------------------------------------------------
print("\n4. Creando Factsheet...")

factsheet = Factsheet(
    general={
        "model_name": "CyberAttack-Classifier",
        "model_type": "RandomForestClassifier",
        "purpose_description": "Clasificación multiclase de tráfico de red para detectar 9 tipos de ataques y tráfico normal",
        "domain_description": "Ciberseguridad / Detección de intrusiones (IDS)",
        "training_data_description": "KDD Cup 1999: Dataset de tráfico de red con ataques DoS, Probe, R2L, U2R",
        "model_information": f"RandomForest con 100 árboles, max_depth=20, {len(feature_cols)} features",
        "target_column": "target",
        "authors": ["Security Team"],
        "contact_information": "security@example.com"
    },
    fairness={
        "protected_feature": "Group",
        "protected_values": [1],  # Grupo de ataques
        "favorable_outcomes": [0]  # Normal = favorable (no es ataque)
    },
    privacy={
        "epsilon": 1.0,
        "quasi_identifiers": ["QI_src_bytes", "QI_dst_bytes"],
        "sensitive_attribute": ["Group"]
    },
    sustainability={
        "use_codecarbon": False,
        "energy_consumed": 0.005,
        "cpu_energy": 0.004,
        "gpu_energy": 0.0,
        "ram_energy": 0.001,
        "emissions": 0.002,
        "duration": 30,
        "country": "Spain",
        "pue": 1.0,
        "wue": 0.0
    },
    save_path="models_and_data/factsheet_cybersecurity.json"
)

print("   Factsheet guardada en: models_and_data/factsheet_cybersecurity.json")

# -------------------------------------------------------------------------
# Paso 5: Crear wrapper y evaluar con TrustEvaluator
# -------------------------------------------------------------------------
print("\n5. Evaluando con TrustEvaluator...")
print("=" * 70)

wrapped_model = ScaledModelWrapper(model, model.scaler_, feature_cols)

evaluator = TrustEvaluator(
    model=wrapped_model,
    train_data=train_df,
    test_data=test_df,
    factsheet=factsheet,
    output_path="models_and_data/trust_evaluation_cybersecurity.json"
)

subset_regre = evaluator.evaluate(show_nan=True)

# Visualizaciones

# evaluator.plot_results()
# evaluator.plot_radar()

print(json.dumps(subset_regre, indent=4))

EVALUACIÓN DE TRUST - CLASIFICACIÓN MULTICLASE DE ATAQUES DE RED

1. Cargando dataset KDD Cup 99...
Descargando KDD Cup 99 dataset...
   Dataset original: 100655 registros
   Después de filtrar clases: 100650 registros
   Después de submuestrear: 30000 registros
   Después de filtrar clases raras: 29976 registros

   Train: 23980 registros
   Test: 5996 registros
   Clases: 3
   Distribución de clases:
      0: normal               -  29017 (96.8%)
      1: neptune_dos          -    254 (0.8%)
      2: smurf_dos            -    705 (2.4%)

2. Entrenando modelo...

Entrenando RandomForest...

3. Evaluando modelo...

Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

      normal       1.00      1.00      1.00      5804
 neptune_dos       1.00      1.00      1.00        51
   smurf_dos       1.00      1.00      1.00       141

    accuracy                           1.00      5996
   macro avg       1.00      1.00      1.00      5996
weigh